# Stage 8 — The Muse Live Skill Validation

**Why this stage:** Stage 7's Adzuna descriptions were ~500-character snippets, which gave 2.0 skills per posting vs 11.5 in training. That is too few to test skill co-occurrence. The Muse public API returns **full HTML job descriptions** and needs no API key.

**Goal:** check whether the Stage 4 association rules from the 2023-2024 LinkedIn data still hold on current postings, and how skill demand has shifted.

**API notes, found by probing the API on 2026-10-14:**
- CloudFront returns HTTP 403 for generic client User-Agents. Requests send a descriptive `User-Agent` that identifies this project.
- The requested category names don't all exist on The Muse:

| Requested | Muse category | Jobs available |
|---|---|---|
| Data Science | Data Science | 2 |
| Engineering | Software Engineering | ~95,000 |
| IT | Computer and IT | ~1,200 |
| Analytics | Data and Analytics | ~15,000 |

- Results are not sorted by date, so publication dates are reported.

Every page is cached in `data/raw/muse_live/pages/`, so **this notebook makes no API calls**. The logic lives in `src/muse_pipeline.py`; to run without the notebook: `python -m src.muse_pipeline`.

## Setup

In [1]:
import json
import logging
import sys
from ast import literal_eval
from pathlib import Path

import pandas as pd


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import muse_pipeline as mp
from src.preprocessing import load_cleaned_jobs

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
paths = mp.muse_paths(PROJECT_ROOT)
pd.set_option("display.max_colwidth", 80)

## Step 1 — Fetch

The fetch covers pages 0-24 (20 jobs each) of each category, 0.5 seconds apart, and stops at a category's last page. Postings are deduplicated by `id`, the `contents` HTML is stripped to plain text with BeautifulSoup, and the requested fields are extracted.

As in Stage 2, **reposted ads** (same company, title, and description) are then dropped.

In [2]:
stats = mp.fetch_all(paths)
postings, total = mp.combine_pages(paths)
raw = mp.extract_postings(postings)
mp.save_raw_csv(raw, paths["raw_csv"])
n_unique = len(raw)
raw, n_reposts = mp.drop_reposts(raw)
log = json.loads(paths["fetch_log"].read_text())
fetch_date = log[-1]["time"][:10]
print(f"API page calls logged: {len(log)}; fetched {total:,}; unique by id {n_unique:,}; "
      f"after dropping {n_reposts} reposts {len(raw):,}")
print(f"Mean description length: {raw['description_words'].mean():,.0f} words")
display(stats)
raw[["id", "name", "company", "locations", "levels", "categories", "publication_date", "description_words"]].head()

INFO src.muse_pipeline: Data Science (Data Science): 1 pages, 2 jobs, last page (1 pages exist)


INFO src.muse_pipeline: Engineering (Software Engineering): 25 pages, 500 jobs, all pages fetched


INFO src.muse_pipeline: IT (Computer and IT): 25 pages, 500 jobs, all pages fetched


INFO src.muse_pipeline: Analytics (Data and Analytics): 25 pages, 500 jobs, all pages fetched


INFO src.muse_pipeline: Fetch: 0 new API calls (76 logged in total)


INFO src.muse_pipeline: 1502 postings fetched, 1502 unique after dedup


INFO src.muse_pipeline: Dropped 124 reposted duplicates (same company, name, description)


API page calls logged: 76; fetched 1,502; unique by id 1,502; after dropping 124 reposts 1,378
Mean description length: 813 words


,category,muse_category,pages,jobs,total,stopped
0,Data Science,Data Science,1,2,2,last page (1 pages exist)
1,Engineering,Software Engineering,25,500,95062,all pages fetched
2,IT,Computer and IT,25,500,1237,all pages fetched
3,Analytics,Data and Analytics,25,500,15191,all pages fetched


,id,name,company,locations,levels,categories,publication_date,description_words
0,5314423,Senior Mulesoft Architect,b atomic,[Flexible / Remote],[Senior Level],"[Data Science, Software Engineer]",2021-02-25T17:17:45Z,596
1,5526783,Data Science Lead,Pilotly,"[Flexible / Remote, Los Angeles, CA, Oakland, CA]","[Mid Level, management]",[Data Science],2021-04-20T21:02:43Z,639
2,22171959,Sr Software Engineer - Android,Uber,"[New York, NY, San Francisco, CA, Sunnyvale, CA]",[Senior Level],[Software Engineering],2026-10-01T22:29:21Z,893
3,22115048,"Sr. Director, Creative Direction - TFT",Riot Games,"[Los Angeles, CA]",[Senior Level],[Software Engineering],2026-09-04T18:30:45Z,1012
4,22123640,Global Markets Risk Sr. Manager,Bank of America,"[New York, NY]",[Senior Level],[Software Engineering],2026-09-09T20:52:33Z,1231


**How current and how diverse is the sample?** Most postings were published in 2026, but a few employers dominate. Their shared template text is a risk for association mining.

In [3]:
pub = pd.to_datetime(raw["publication_date"])
print(f"Published {pub.min():%Y-%m-%d} to {pub.max():%Y-%m-%d}; {(pub.dt.year == 2026).mean():.0%} in 2026; "
      f"{(pub >= pd.Timestamp(fetch_date, tz='UTC') - pd.Timedelta(days=90)).mean():.0%} in the last 90 days")
print(f"{raw['company'].nunique()} companies")
raw["company"].value_counts().head(8).to_frame("postings").assign(share=lambda d: (d["postings"] / len(raw)).round(3))

Published 2021-02-25 to 2026-10-13; 85% in 2026; 73% in the last 90 days
115 companies


,postings,share
company,,
SpaceX,467,0.339
Bank of America,168,0.122
Jabil,77,0.056
Uber,73,0.053
Bechtel,71,0.052
Celonis,54,0.039
Riot Games,34,0.025
EPAM Systems,28,0.020


## Step 2 — Preprocess

- **Skills:** the same Stage 2 regex matcher and 245-skill vocabulary, run on the **full** plain-text description.
- **Experience level:** from The Muse `levels`. Internship → Internship, Entry Level → Entry, Mid Level → Mid-Senior, Senior Level → Senior, Manager → Director, Executive → Executive, otherwise Unknown.
- **State:** best effort, from the first location that names a state.

There is no salary in The Muse data, so no salary prediction is attempted.

In [4]:
vocab = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "skill_vocabulary.csv")
vocab["forms"] = vocab["forms"].apply(literal_eval)
muse = mp.preprocess(raw, vocab, fetch_date)
muse.to_csv(paths["cleaned"], index=False)
muse = mp.load_muse_cleaned(paths["cleaned"])
train = load_cleaned_jobs(PROJECT_ROOT / "data" / "processed" / "cleaned_jobs.csv")
adzuna = mp.adzuna_reference(PROJECT_ROOT)
print(f"Skills per posting: Muse {muse['n_skills'].mean():.1f} vs Adzuna {adzuna['skills']:.1f} "
      f"vs training {train['matched_skills'].str.len().mean():.1f}")
print(f"Description words: Muse {raw['description_words'].mean():,.0f} vs Adzuna {adzuna['words']:,.0f}")
print("Experience levels:", muse["experience_level"].value_counts().to_dict())
print(f"State parsed for {muse['state'].notna().mean():.0%}")
muse.head()

INFO src.preprocessing: Matched skills: 51 nested phrases matched first (time management, project management, inventory management, written communication, verbal communication, team management, team leadership, food safety, risk management, patient education, business development, professional development, budget management, case management, performance management, ability to work in a fastpaced environment, regulatory compliance, continuing education, electrical engineering, financial reporting, selfmotivation, account management, strategic planning, civil engineering, change management, construction management, mechanical engineering, team collaboration, training and development, customer service skills, retail management, technical writing, rn license, time management skills, financial management, travel nursing, report writing, patient assessment, risk assessment, english proficiency, stress management, treatment planning, equipment maintenance, oral communication, ability to work 

INFO src.preprocessing: Matched description skills: 1374 of 1378 jobs have at least one, mean 16.2 per job


Skills per posting: Muse 16.2 vs Adzuna 2.0 vs training 11.5
Description words: Muse 813 vs Adzuna 71
Experience levels: {'Senior': 787, 'Mid-Senior': 565, 'Internship': 15, 'Entry': 11}
State parsed for 71%


,id,job_title,company_name,location,state,experience_level,categories,matched_skills,n_skills,fetch_date,publication_date
0,5314423,senior mulesoft architect,b atomic,Flexible / Remote,NaN,Senior,Data Science; Software Engineer,"[problem solving, leadership, organization, documentation, management, engin...",15,2026-10-14,2021-02-25
1,5526783,data science lead,Pilotly,"Flexible / Remote; Los Angeles, CA; Oakland, CA",CA,Mid-Senior,Data Science,"[communication, organization, data analysis, reporting, research, python, sq...",10,2026-10-14,2021-04-20
2,22171959,sr software engineer - android,Uber,"New York, NY; San Francisco, CA; Sunnyvale, CA",NY,Senior,Software Engineering,"[leadership, collaboration, excel, research, management, engineering, develo...",12,2026-10-14,2026-10-01
3,22115048,"sr. director, creative direction - tft",Riot Games,"Los Angeles, CA",CA,Senior,Software Engineering,"[communication, leadership, organization, collaboration, decision making, sa...",17,2026-10-14,2026-09-04
4,22123640,global markets risk sr. manager,Bank of America,"New York, NY",NY,Senior,Software Engineering,"[communication, problem solving, teamwork, leadership, organization, collabo...",28,2026-10-14,2026-09-09


## Step 3 — Load into the warehouse

`muse_job_postings` in `skillmap.db` has one row per posting. `matched_skills` is stored as a JSON array, and `source` is 'muse_live'. The table is dropped and recreated on each run.

In [5]:
warehouse = mp.load_into_warehouse(muse, paths["db"])
print(f"SELECT COUNT(*), AVG(n_skills) FROM muse_job_postings -> {warehouse['rows']:,}, {warehouse['avg_skills']:.2f}")

INFO src.muse_pipeline: muse_job_postings: 1378 rows, mean 16.20 skills


SELECT COUNT(*), AVG(n_skills) FROM muse_job_postings -> 1,378, 16.20


## Step 4 — Association rule validation

Postings with at least 2 skills are mined with Apriori: `min_support = 0.05` (or 0.02 if that gives fewer than 10 rules), confidence > 0.5, lift > 1.5. Each of the Stage 4 top-20 training rules is then checked:
- `validated`: the rule is among the Muse rules
- `lift_holds`: confidence > 0.5 and lift > 1.5 on Muse data, ignoring support
- `muse_max_lift` = 1 / the consequent's share of postings, the highest lift the rule *could* reach. When a consequent appears in most postings, lift is capped near 1 even if the skills always co-occur.
- **Company-capped sensitivity:** the check is repeated with at most 25 postings per company, so a few employers' boilerplate can't drive the result.

In [6]:
onehot, rules, support, n_qualify = mp.mine_muse_rules(muse)
rules.to_csv(OUTPUT_DIR / "08_muse_association_rules.csv", index=False)
capped_onehot, capped_rules, capped_support, capped_n = mp.mine_muse_rules(mp.cap_per_company(muse))
validation = mp.validate_rules(pd.read_csv(OUTPUT_DIR / "04_association_rules.csv"), rules, onehot,
                               capped_rules, capped_onehot)
validation.to_csv(OUTPUT_DIR / "08_rules_validation.csv", index=False)
print(f"{n_qualify:,} postings with >= {mp.MIN_SKILLS} skills; min_support={support}; {len(rules):,} Muse rules")
print(f"Validated: {validation['validated'].sum()} of {len(validation)}; lift holds ignoring support: "
      f"{validation['lift_holds'].sum()}; company-capped ({capped_n} postings): "
      f"{validation['validated_company_capped'].sum()}")
validation[["rule", "training_lift", "muse_lift", "muse_max_lift", "training_confidence", "muse_confidence",
            "validated", "validated_company_capped"]]

INFO src.association: Apriori (min_support=0.05) on 1372 transactions: 4165 frequent itemsets


INFO src.association: Rules with confidence > 0.50 and lift > 1.50: 20599


INFO src.association: Apriori (min_support=0.05) on 601 transactions: 2590 frequent itemsets


INFO src.association: Rules with confidence > 0.50 and lift > 1.50: 2580


1,372 postings with >= 2 skills; min_support=0.05; 20,599 Muse rules
Validated: 2 of 20; lift holds ignoring support: 4; company-capped (601 postings): 1


,rule,training_lift,muse_lift,muse_max_lift,training_confidence,muse_confidence,validated,validated_company_capped
0,powerpoint -> excel,5.4205,5.8299,6.3226,0.9201,0.9221,True,False
1,"development, management, organization -> leadership",2.1662,1.5830,2.6589,0.5330,0.5953,True,False
2,accounting -> finance,5.5405,6.5254,8.4691,0.5640,0.7705,False,False
3,marketing -> sales,2.5168,5.8023,9.2081,0.5350,0.6301,False,True
4,finance -> accounting,5.5405,6.5254,22.4918,0.5361,0.2901,False,False
5,"compliance, leadership -> management",1.6851,1.4855,1.8245,0.7610,0.8142,False,False
6,interpersonal skills -> communication,1.7413,1.3859,2.3098,0.6632,0.6000,False,False
7,"education, leadership -> development",1.6883,1.3686,1.6333,0.6209,0.8379,False,False
8,"collaboration, management -> development",1.7523,1.3176,1.6333,0.6444,0.8067,False,False
9,"leadership, management, organization -> development",1.7542,1.3149,1.6333,0.6451,0.8050,False,False


The highest-lift Muse rules are long itemsets at exactly the same support. That is one employer's template text repeated across its postings. For example, {driving, leadership, risk management} → … comes from Bank of America postings:

In [7]:
display(rules.head(5)[["antecedents", "consequents", "support", "confidence", "lift"]])
template = {"driving", "leadership", "risk management", "collaboration", "development", "education", "flexibility", "management"}
muse.loc[muse["matched_skills"].map(lambda s: template <= set(s)), "company_name"].value_counts().head()

,antecedents,consequents,support,confidence,lift
0,"collaboration, development, education, flexibility, management","driving, innovation, leadership, risk management",0.051,0.6604,10.9161
1,"driving, innovation, leadership, risk management","collaboration, development, education, flexibility, management",0.051,0.8434,10.9161
2,"development, driving, innovation, leadership, risk management","collaboration, education, flexibility, management",0.051,0.8537,10.7451
3,"collaboration, education, flexibility, management","development, driving, innovation, leadership, risk management",0.051,0.6422,10.7451
4,"development, driving, leadership, risk management","collaboration, education, flexibility, innovation, management",0.051,0.7778,10.6711


company_name
Bank of America    74
Vanguard            1
Name: count, dtype: int64

## Step 5 — Skill demand: Muse vs training

Each skill's share of postings is compared with all training jobs and with **training tech jobs**, the Stage 6 *Data, Tech & Engineering* cluster. That cluster is the closest counterpart to The Muse tech categories. Shifts mix real demand change with source differences (The Muse vs LinkedIn, description length and style).

In [8]:
labels = pd.read_csv(OUTPUT_DIR / "06_cluster_labels.csv")
tech_ids = set(labels.loc[labels["kmeans_cluster_name"] == "Data, Tech & Engineering", "job_id"])
comparison = mp.skill_comparison(muse, train, tech_ids)
comparison.to_csv(OUTPUT_DIR / "08_skill_comparison.csv", index=False)
cols = ["skill", "muse_share", "train_tech_share", "train_share", "change_vs_tech_pp"]
print("Top 20 skills in Muse data:")
display(comparison.head(20)[cols])
print("Most common technical skills:")
comparison[comparison["is_technical"]].head(10)[cols]

Top 20 skills in Muse data:


,skill,muse_share,train_tech_share,train_share,change_vs_tech_pp
0,engineering,0.7010,0.6808,0.1698,2.0186
1,design,0.6219,0.7973,0.2090,-17.5341
2,development,0.6096,0.7411,0.3678,-13.1471
3,management,0.5464,0.5000,0.4516,4.6444
4,education,0.4862,0.3454,0.3673,14.0779
5,communication,0.4311,0.3809,0.3809,5.0163
6,security,0.3759,0.2747,0.1401,10.1233
7,leadership,0.3745,0.2772,0.2460,9.7273
8,bachelor's degree,0.3418,0.2148,0.1305,12.7012
9,python,0.3179,0.1756,0.0391,14.2208


Most common technical skills:


,skill,muse_share,train_tech_share,train_share,change_vs_tech_pp
9,python,0.3179,0.1756,0.0391,14.2208
23,sql,0.2134,0.1501,0.0418,6.3303
34,c++,0.1161,0.0584,0.0119,5.7730
39,ci/cd,0.1089,0.0719,0.0139,3.6923
42,agile,0.1052,0.1718,0.0480,-6.6572
44,linux,0.1009,0.0652,0.0147,3.5632
49,kubernetes,0.0820,0.0492,0.0097,3.2823
50,java,0.0806,0.0910,0.0185,-1.0449
55,aws,0.0740,0.1173,0.0257,-4.3243
57,machine learning,0.0711,0.0751,0.0172,-0.3991


In [9]:
rises, drops = mp.demand_shifts(comparison)
print("Largest rises vs training tech jobs:")
display(rises[cols])
print("Largest drops:")
drops[cols]

Largest rises vs training tech jobs:


,skill,muse_share,train_tech_share,train_share,change_vs_tech_pp
13,manufacturing,0.3019,0.1536,0.0880,14.8324
9,python,0.3179,0.1756,0.0391,14.2208
4,education,0.4862,0.3454,0.3673,14.0779


Largest drops:


,skill,muse_share,train_tech_share,train_share,change_vs_tech_pp
1,design,0.6219,0.7973,0.2090,-17.5341
2,development,0.6096,0.7411,0.3678,-13.1471
80,healthcare,0.0559,0.1477,0.1969,-9.1830


## Step 6 — Summary

In [10]:
summary = mp.build_summary(fetch_date, stats, total, n_unique, n_reposts, raw, muse, train, adzuna, warehouse,
                           rules, support, n_qualify, validation, comparison, len(log),
                           {"n": capped_n, "support": capped_support, "n_rules": len(capped_rules)})
(OUTPUT_DIR / "08_muse_summary.txt").write_text(summary, encoding="utf-8")
print(summary)

SkillMap — Stage 8 The Muse Live Skill Validation Summary
Generated: 2026-10-14 12:55:12
Fetch date: 2026-10-14
Source: https://www.themuse.com/api/public/jobs (no API key; descriptive User-Agent). API page calls logged: 76

Fetch
    category        muse_category  pages  jobs  total                   stopped
Data Science         Data Science      1     2      2 last page (1 pages exist)
 Engineering Software Engineering     25   500  95062         all pages fetched
          IT      Computer and IT     25   500   1237         all pages fetched
   Analytics   Data and Analytics     25   500  15191         all pages fetched
Requested categories are mapped to The Muse names; 'Data Science' exists but has only 2 jobs.
Total postings fetched: 1,502; unique after dedup by id: 1,502; after dropping 124 reposted ads (same company, title, description): 1,378
Employers: 115 companies; largest: SpaceX 467 (34%), Bank of America 168 (12%), Jabil 77 (6%), Uber 73 (5%), Bechtel 71 (5%)
Publication 

## Findings

- **Full descriptions fix the Stage 7 problem.**
  - Muse postings average **813 words vs 71** for Adzuna snippets, and **16.2 skills per posting** vs 2.0 (Adzuna) and 11.5 (training).
  - After deduplication (124 reposts dropped) there are 1,378 unique postings. 73% were published in the 90 days before the fetch.
- **Only 2 of the 20 top training rules are confirmed as rules: *powerpoint → excel* (lift 5.4 → 5.8) and *development, management, organization → leadership*.** With at most 25 postings per company, only *marketing → sales* passes.
- **Most failures are a base-rate effect, not broken patterns.**
  - In tech postings, *development*, *design*, and *management* appear in 55-62% of postings, so lift can't exceed about 1.6-1.8.
  - 14 of the 16 failing rules hit this ceiling, and **all 14 have higher confidence on Muse than in training**. For example, *engineering → design* rises from 0.55 to 0.71, and *education, leadership → development* from 0.62 to 0.84.
  - The skills still co-occur; they are just no longer distinctive in a tech-only sample.
  - *accounting ↔ finance* and *marketing → sales* keep high lift (5.8-6.5), but fall under 5% support.
- **The sample is concentrated.** SpaceX is 34% of postings and Bank of America 12%, out of 115 companies. Their template text produces the highest-lift Muse rules, so results are also reported company-capped.
- **Skill demand vs training tech jobs.**
  - *python* rose (32% vs 18%), as did *sql*, *c++*, *ci/cd*, *linux*, and *kubernetes*.
  - *agile* (11% vs 17%) and *aws* (7% vs 12%) fell.
  - *manufacturing* rose and *healthcare* fell, which reflects the employer mix (SpaceX, Jabil), not the market.
- **Answer.** The core skill groupings from 2023-2024 still hold in 2026 tech postings: confidence is stable or higher. But lift-based rules mined on all industries don't transfer to a tech-only sample, because the base rates differ. Validating them properly would need a source with a similar industry mix.